# US Recession 2025 Forecast - Model Comparison

This notebook compares the v1 and v2 recession forecast models.

## Objectives
1. Compare v1 and v2 predictions on the same data
2. Analyze parameter sensitivity
3. Visualize feature importance
4. Understand the impact of temporal decay

In [ ]:
# Add project root to path
import sys
from pathlib import Path

# Add project root (3 levels up from this notebook)
project_root = Path().absolute().parent.parent.parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported")

In [ ]:
# Import both v1 and v2 models
from forecasts.us_recession_2025.model import RecessionModel
from forecasts.us_recession_2025.model_v2 import RecessionModelV2

# Initialize models
model_v1 = RecessionModel()
model_v2 = RecessionModelV2()

print(f"✓ Initialized {model_v1.get_name()}")
print(f"✓ Initialized {model_v2.get_name()}")

## 1. Model Comparison: V1 vs V2

Let's compare the predictions from both models using the same current data.

In [ ]:
# Calculate probabilities using default parameters
print("Calculating v1 probability...")
prob_v1 = model_v1.calculate_probability()

print("Calculating v2 probability...")
prob_v2 = model_v2.calculate_probability()

print(f"\nV1 Probability: {prob_v1:.4f} ({prob_v1*100:.2f}%)")
print(f"V2 Adjusted Probability: {prob_v2:.4f} ({prob_v2*100:.2f}%)")

In [ ]:
# Get detailed breakdown from v2 model
breakdown = model_v2.get_probability_breakdown()

print("V2 Model Breakdown:")
print(f"  Base Probability: {breakdown['base_probability']:.4f}")
print(f"  Adjusted Probability: {breakdown['adjusted_probability']:.4f}")
print(f"  Days Remaining: {breakdown['days_remaining']}")
print(f"\nTemporal Metadata:")
for key, value in breakdown['temporal_metadata'].items():
    print(f"  {key}: {value}")

In [ ]:
# Create side-by-side comparison visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

models = ['V1', 'V2\n(Base)', 'V2\n(Adjusted)']
probabilities = [prob_v1, breakdown['base_probability'], breakdown['adjusted_probability']]
colors = ['#4ecdc4', '#45b7d1', '#ff6b6b']

bars = ax.bar(models, [p * 100 for p in probabilities], color=colors, alpha=0.7, edgecolor='black')

# Add value labels on bars
for bar, prob in zip(bars, probabilities):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{prob*100:.2f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Recession Probability (%)', fontsize=12)
ax.set_title('Model Comparison: V1 vs V2', fontsize=14, fontweight='bold')
ax.set_ylim(0, max([p * 100 for p in probabilities]) * 1.2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDifference (V1 - V2 Adjusted): {(prob_v1 - breakdown['adjusted_probability'])*100:.2f} percentage points")

In [ ]:
# Display indicator signals from v2 model
print("Indicator Signals (V2 Model):")
print("=" * 50)
for indicator, signal in breakdown['indicator_signals'].items():
    print(f"{indicator:25s}: {signal:.4f} ({signal*100:.2f}%)")

# Visualize indicator signals
fig, ax = plt.subplots(figsize=(10, 6))

indicators = list(breakdown['indicator_signals'].keys())
signals = [breakdown['indicator_signals'][ind] * 100 for ind in indicators]

# Create horizontal bar chart
bars = ax.barh(indicators, signals, color='steelblue', alpha=0.7, edgecolor='black')

# Add value labels
for bar, signal in zip(bars, signals):
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2.,
            f'{signal:.2f}%',
            ha='left', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Signal Strength (%)', fontsize=12)
ax.set_title('Individual Indicator Signals', fontsize=14, fontweight='bold')
ax.set_xlim(0, 100)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Parameter Sensitivity Analysis

Let's analyze how different parameter values affect the model's predictions.

In [ ]:
# Test sensitivity to yield curve weight
print("Testing sensitivity to yield curve weight...")

weight_values = np.linspace(0.0, 1.0, 11)
probabilities_v1 = []
probabilities_v2_base = []
probabilities_v2_adjusted = []

for weight in weight_values:
    params = {'yield_curve_weight': weight}
    
    # V1 model
    prob_v1_test = model_v1.calculate_probability(params)
    probabilities_v1.append(prob_v1_test)
    
    # V2 model
    prob_v2_test = model_v2.calculate_probability(params)
    breakdown_test = model_v2.get_probability_breakdown(params)
    probabilities_v2_base.append(breakdown_test['base_probability'])
    probabilities_v2_adjusted.append(breakdown_test['adjusted_probability'])

print("✓ Sensitivity analysis complete")

In [ ]:
# Visualize sensitivity to yield curve weight
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(weight_values, [p * 100 for p in probabilities_v1], 
        marker='o', label='V1', linewidth=2, color='#4ecdc4')
ax.plot(weight_values, [p * 100 for p in probabilities_v2_base], 
        marker='s', label='V2 (Base)', linewidth=2, color='#45b7d1')
ax.plot(weight_values, [p * 100 for p in probabilities_v2_adjusted], 
        marker='^', label='V2 (Adjusted)', linewidth=2, color='#ff6b6b')

ax.set_xlabel('Yield Curve Weight', fontsize=12)
ax.set_ylabel('Recession Probability (%)', fontsize=12)
ax.set_title('Parameter Sensitivity: Yield Curve Weight', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate sensitivity (change in probability per unit change in weight)
v1_sensitivity = (probabilities_v1[-1] - probabilities_v1[0]) * 100
v2_base_sensitivity = (probabilities_v2_base[-1] - probabilities_v2_base[0]) * 100
v2_adj_sensitivity = (probabilities_v2_adjusted[-1] - probabilities_v2_adjusted[0]) * 100

print(f"\nSensitivity (change from weight=0 to weight=1):")
print(f"  V1: {v1_sensitivity:.2f} percentage points")
print(f"  V2 (Base): {v2_base_sensitivity:.2f} percentage points")
print(f"  V2 (Adjusted): {v2_adj_sensitivity:.2f} percentage points")

In [ ]:
# Test sensitivity to all weight parameters
print("Testing sensitivity to all weight parameters...")

weight_params = [
    'yield_curve_weight',
    'unemployment_weight',
    'gdp_weight',
    'confidence_weight',
    'leading_indicators_weight'
]

sensitivities = {}

for param in weight_params:
    probs = []
    for weight in [0.0, 0.5, 1.0]:
        params = {param: weight}
        prob = model_v2.calculate_probability(params)
        probs.append(prob)
    
    # Calculate sensitivity as range
    sensitivities[param] = (max(probs) - min(probs)) * 100

print("\nParameter Sensitivities (range in percentage points):")
for param, sensitivity in sorted(sensitivities.items(), key=lambda x: x[1], reverse=True):
    print(f"  {param:30s}: {sensitivity:.2f}")

In [ ]:
# Visualize parameter sensitivities
fig, ax = plt.subplots(figsize=(10, 6))

params_sorted = sorted(sensitivities.items(), key=lambda x: x[1], reverse=True)
param_names = [p[0].replace('_weight', '').replace('_', ' ').title() for p, _ in params_sorted]
param_values = [v for _, v in params_sorted]

bars = ax.barh(param_names, param_values, color='coral', alpha=0.7, edgecolor='black')

# Add value labels
for bar, value in zip(bars, param_values):
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2.,
            f'{value:.2f}',
            ha='left', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Sensitivity (percentage point range)', fontsize=12)
ax.set_title('Parameter Sensitivity Analysis', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Identify most influential parameter
most_influential = max(sensitivities.items(), key=lambda x: x[1])
print(f"\nMost influential parameter: {most_influential[0]} (range: {most_influential[1]:.2f} pp)")

## 3. Feature Importance Analysis

Let's analyze the contribution of each feature in the v2 model.

In [ ]:
# Display engineered features from v2 model
print("Engineered Features (V2 Model):")
print("=" * 60)

if breakdown['features']:
    for feature, value in sorted(breakdown['features'].items()):
        if value is not None:
            print(f"{feature:40s}: {value:10.4f}")
        else:
            print(f"{feature:40s}: None (insufficient data)")
else:
    print("No engineered features available")

# Count available features
available_features = sum(1 for v in breakdown['features'].values() if v is not None)
total_features = len(breakdown['features'])
print(f"\nAvailable features: {available_features}/{total_features}")

In [ ]:
# Compare v1 indicators with v2 indicators and features
print("Data Comparison: V1 vs V2")
print("=" * 60)

print("\nV1 Indicators (6 total):")
v1_indicators = [
    'yield_curve',
    'unemployment',
    'gdp',
    'consumer_confidence',
    'leading_indicators',
    'jobless_claims'
]
for ind in v1_indicators:
    print(f"  - {ind}")

print(f"\nV2 Indicators ({len(breakdown['indicators'])} total):")
for ind in sorted(breakdown['indicators'].keys()):
    print(f"  - {ind}")

print(f"\nV2 Engineered Features ({available_features} available):")
for feature in sorted(breakdown['features'].keys()):
    status = "✓" if breakdown['features'][feature] is not None else "✗"
    print(f"  {status} {feature}")

# Calculate data expansion
v2_total = len(breakdown['indicators']) + available_features
expansion_factor = v2_total / len(v1_indicators)
print(f"\nData expansion: {expansion_factor:.1f}x (from {len(v1_indicators)} to {v2_total} data points)")

In [ ]:
# Categorize and visualize features
feature_categories = {
    'Rate of Change': [],
    'Moving Averages': [],
    'Volatility': [],
    'Other': []
}

for feature in breakdown['features'].keys():
    if 'roc' in feature:
        feature_categories['Rate of Change'].append(feature)
    elif 'ma' in feature:
        feature_categories['Moving Averages'].append(feature)
    elif 'volatility' in feature or 'vol' in feature:
        feature_categories['Volatility'].append(feature)
    else:
        feature_categories['Other'].append(feature)

# Count features by category
category_counts = {cat: len(features) for cat, features in feature_categories.items() if features}

# Visualize feature categories
fig, ax = plt.subplots(figsize=(10, 6))

categories = list(category_counts.keys())
counts = list(category_counts.values())
colors_cat = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#95E1D3']

wedges, texts, autotexts = ax.pie(counts, labels=categories, autopct='%1.1f%%',
                                    colors=colors_cat[:len(categories)],
                                    startangle=90, textprops={'fontsize': 11})

# Make percentage text bold
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title('Feature Categories in V2 Model', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nFeature Categories:")
for cat, features in feature_categories.items():
    if features:
        print(f"\n{cat} ({len(features)}):")
        for f in features:
            print(f"  - {f}")

## 4. Temporal Decay Analysis

Let's analyze the impact of temporal decay on the v2 model's predictions.

In [ ]:
# Calculate v2 probability without temporal decay
prob_v2_no_decay = model_v2.calculate_probability(apply_temporal_decay=False)
prob_v2_with_decay = model_v2.calculate_probability(apply_temporal_decay=True)

print("Temporal Decay Impact:")
print("=" * 50)
print(f"Without temporal decay: {prob_v2_no_decay:.4f} ({prob_v2_no_decay*100:.2f}%)")
print(f"With temporal decay:    {prob_v2_with_decay:.4f} ({prob_v2_with_decay*100:.2f}%)")
print(f"Difference:             {(prob_v2_no_decay - prob_v2_with_decay):.4f} ({(prob_v2_no_decay - prob_v2_with_decay)*100:.2f} pp)")
print(f"\nDays remaining: {breakdown['days_remaining']}")
print(f"Decay method: {breakdown['temporal_metadata'].get('method', 'N/A')}")

In [ ]:
# Visualize temporal decay impact
fig, ax = plt.subplots(figsize=(10, 6))

models_decay = ['Base\n(No Decay)', 'Adjusted\n(With Decay)']
probs_decay = [prob_v2_no_decay, prob_v2_with_decay]
colors_decay = ['#45b7d1', '#ff6b6b']

bars = ax.bar(models_decay, [p * 100 for p in probs_decay], 
              color=colors_decay, alpha=0.7, edgecolor='black', width=0.6)

# Add value labels
for bar, prob in zip(bars, probs_decay):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{prob*100:.2f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Add arrow showing the adjustment
ax.annotate('', xy=(1, prob_v2_with_decay * 100), xytext=(0, prob_v2_no_decay * 100),
            arrowprops=dict(arrowstyle='->', lw=2, color='red'))

ax.set_ylabel('Recession Probability (%)', fontsize=12)
ax.set_title('Impact of Temporal Decay Adjustment', fontsize=14, fontweight='bold')
ax.set_ylim(0, max([p * 100 for p in probs_decay]) * 1.2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Summary and Conclusions

### Key Findings:

1. **Model Comparison**: The v2 model provides both base and adjusted probabilities, with temporal decay accounting for time-to-deadline effects.

2. **Parameter Sensitivity**: Different weight parameters have varying impacts on the final probability. The most influential parameters should be carefully calibrated.

3. **Feature Engineering**: The v2 model incorporates additional indicators and engineered features, providing a richer data foundation for predictions.

4. **Temporal Decay**: The temporal adjustment modifies the base probability based on time remaining, reflecting the principle that probabilities should adjust as deadlines approach.

### Next Steps:

- Conduct backtesting to evaluate historical performance
- Calibrate temporal decay parameters using historical data
- Analyze feature importance through correlation with actual outcomes
- Consider ensemble approaches combining v1 and v2 predictions

In [ ]:
# Create comprehensive comparison table
comparison_data = {
    'Metric': [
        'Model Version',
        'Number of Indicators',
        'Engineered Features',
        'Temporal Decay',
        'Current Probability',
        'Base Probability',
        'Adjusted Probability'
    ],
    'V1': [
        'v1',
        '6',
        'No',
        'No',
        f'{prob_v1*100:.2f}%',
        'N/A',
        'N/A'
    ],
    'V2': [
        'v2',
        str(len(breakdown['indicators'])),
        f'Yes ({available_features})',
        'Yes',
        f'{prob_v2*100:.2f}%',
        f"{breakdown['base_probability']*100:.2f}%",
        f"{breakdown['adjusted_probability']*100:.2f}%"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison Summary:")
print("=" * 70)
print(comparison_df.to_string(index=False))

print(f"\n✓ Analysis complete")
print(f"✓ Notebook execution finished at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")